# 02 — Is block identity decodable from a slice?

The whole leakage story assumes a model can recognise *which scan* a slice came from. That has
never been tested; the original thesis inferred it from Grad-CAM heatmaps, which is not evidence.

**Design.** Within one class, hold out 2 slices per block. Train a K-way block classifier on the
rest. Chance is 1/K. Then repeat with **randomly permuted block labels** as a null control — the
null must land at chance, or the probe is broken.

High probe accuracy ⇒ identity is trivially decodable ⇒ the leakage mechanism is demonstrated
directly. Near-chance accuracy ⇒ the mechanism story collapses and must be rewritten.

Budget: ~1 GPU-h.

In [ ]:
# ── environment ──────────────────────────────────────────────────────────
# 1. Add the dataset:  hamdallak/the-iqothnccd-lung-cancer-dataset
# 2. Add your uploaded library dataset: iqoth-honest-lib  (upload src/iqoth/)
import sys, os, json, time, types

print("Mounted in /kaggle/input:", os.listdir("/kaggle/input") if os.path.isdir("/kaggle/input") else "no /kaggle/input")

LIB = None
ROOT = None
if os.path.isdir("/kaggle/input"):
    for root, dirs, files in os.walk("/kaggle/input"):
        if "splits.py" in files and "audit.py" in files and not LIB:
            LIB = root
        if any("bengin" in d.lower() or "benign" in d.lower() for d in dirs) and not ROOT:
            ROOT = root

if LIB:
    pkg = types.ModuleType("iqoth")
    pkg.__path__ = [LIB]
    sys.modules["iqoth"] = pkg
    sys.path.insert(0, LIB)
else:
    sys.path.insert(0, "src")          # local fallback

if not ROOT:
    ROOT = os.environ.get("IQOTH_ROOT")

assert os.path.isdir(ROOT or ""), f"dataset not found; /kaggle/input tree: {list(os.walk('/kaggle/input')) if os.path.isdir('/kaggle/input') else 'no input'}"

RESULTS = "/kaggle/working/results"
os.makedirs(RESULTS, exist_ok=True)

from iqoth import splits, audit, metrics
print("lib:", LIB)
print("dataset:", ROOT)

import torch
from iqoth import engine, models, data
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
import numpy as np, torch.nn as nn
from collections import defaultdict
from torch.utils.data import DataLoader

recs = splits.build_registry(ROOT)
OUT = f"{RESULTS}/02_identity"; os.makedirs(OUT, exist_ok=True)

def probe(class_name, seed=42, shuffle_labels=False, epochs=15, holdout=2):
    data.set_seed(seed)
    rs = sorted([r for r in recs if r["class_name"] == class_name], key=lambda r: r["pos"])
    bids = sorted({r["block_id"] for r in rs}, key=lambda b: int(b.split("_")[1]))
    bid2y = {b: i for i, b in enumerate(bids)}

    if shuffle_labels:
        # Null control: keep the same label multiset, destroy the association.
        rng = np.random.default_rng(seed)
        perm = list(bid2y.values()); rng.shuffle(perm)
        assign = dict(zip(bids, perm))
        for r in rs: r["_y"] = assign[r["block_id"]]
        groups = defaultdict(list)
        for r in rs: groups[r["_y"]].append(r)
    else:
        for r in rs: r["_y"] = bid2y[r["block_id"]]
        groups = defaultdict(list)
        for r in rs: groups[r["_y"]].append(r)

    tr, te = [], []
    for y, items in groups.items():
        rng = np.random.default_rng(seed + y)
        idx = rng.permutation(len(items))
        for j, i in enumerate(idx):
            (te if j < holdout else tr).append(items[i])

    tf_tr = data.build_transforms("plain", 224, True)
    tf_te = data.build_transforms("plain", 224, False)
    class DS(torch.utils.data.Dataset):
        def __init__(s, items, tf): s.items, s.tf = items, tf
        def __len__(s): return len(s.items)
        def __getitem__(s, i):
            from PIL import Image
            r = s.items[i]
            return s.tf(Image.open(r["image_path"]).convert("RGB")), r["_y"]

    K = len(bids)
    dl_tr = DataLoader(DS(tr, tf_tr), batch_size=32, shuffle=True, num_workers=2)
    dl_te = DataLoader(DS(te, tf_te), batch_size=32, shuffle=False, num_workers=2)

    import torchvision.models as tvm
    m = tvm.resnet18(weights="DEFAULT")
    m.fc = nn.Linear(m.fc.in_features, K)
    dev = engine.get_device(); m.to(dev)
    opt = torch.optim.AdamW(m.parameters(), lr=3e-4, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    for ep in range(epochs):
        m.train()
        for x, y in dl_tr:
            x, y = x.to(dev), y.to(dev)
            opt.zero_grad(); crit(m(x), y).backward(); opt.step()
    m.eval(); ok = n = 0
    with torch.no_grad():
        for x, y in dl_te:
            ok += (m(x.to(dev)).argmax(1).cpu() == y).sum().item(); n += len(y)
    return {"class": class_name, "K": K, "chance": 100 / K, "accuracy": 100 * ok / n,
            "n_test": n, "shuffled_labels": shuffle_labels}

In [ ]:
out = []
for cname in ["Bengin cases", "Normal cases", "Malignant cases"]:
    for sh in (False, True):
        r = probe(cname, seed=42, shuffle_labels=sh)
        lift = r["accuracy"] / r["chance"]
        print(f"{cname:<18} {'NULL' if sh else 'real':<5} K={r['K']:>3} "
              f"chance={r['chance']:5.2f}%  acc={r['accuracy']:6.2f}%  lift={lift:5.1f}x")
        out.append({**r, "lift": lift})
with open(f"{OUT}/identity_probe.json", "w") as f:
    json.dump(out, f, indent=2)

In [ ]:
real = [o for o in out if not o["shuffled_labels"]]
null = [o for o in out if o["shuffled_labels"]]
print("\nVERDICT")
for r, n in zip(real, null):
    if r["lift"] > 3 and n["lift"] < 2:
        print(f"  {r['class']}: identity IS decodable ({r['lift']:.1f}x chance, "
              f"null {n['lift']:.1f}x) -> leakage mechanism demonstrated")
    else:
        print(f"  {r['class']}: NOT clearly decodable ({r['lift']:.1f}x, null {n['lift']:.1f}x) "
              f"-> weaken the mechanism claim in the paper")